In [16]:
import os
base_path =os.path.abspath("")
base_path = base_path + "\\"
header_path = base_path + 'headers\\header_data_treatment.ipynb'
header_path

'C:\\Users\\raphael.wargon\\Documents\\GitHub\\sorting_univ\\script\\headers\\header_data_treatment.ipynb'

In [17]:
%run $header_path

default
initalizing spark nlp session w gpu


In [3]:
sc = spark.sparkContext
#works_au_af_path = 'D:\\openalex-snapshot\\data_extracted\\works_au_af.parquet'
#works_path = 'D:\\openalex-snapshot\\data_extracted\\works.parquet'


In [18]:
df_works =(spark.read.format('parquet').load("file:\\" + main_path_openalex + 'works.parquet')
            .select(func.col('id').alias('work_id') , "funders",'awards'))

In [19]:
df_works.show(truncate = False)

+--------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+
|work_id                         |funders                                                                                                                                                                                                                                                                      |awards|
+--------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+
|https://openalex.org/W1995237132|[{"id":"https://openalex.org/F

In [23]:
df_works.filter(func.size(func.col('awards')) >0).show(truncate= False)

+--------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [20]:
df_works.printSchema()

root
 |-- work_id: string (nullable = true)
 |-- funders: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- awards: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [28]:
schema_funder = StructType([StructField('id',StringType(),True),  
StructField('display_name',StringType(),True),  
StructField('ror',StringType(),True)])

schema_awards = StructType([StructField('id',StringType(),True),  
StructField('funder_award_id',StringType(),True),  
StructField('funder_id',StringType(),True),  
StructField('funder_display_name',StringType(),True)])

works_funders = (df_works
                 .withColumn('funders', func.explode(func.col('funders')))
                 .withColumn('funders', func.from_json(func.col('funders'), schema= schema_funder))
                 .withColumn('funder_id', func.col('funders.id'))
                 .withColumn('awards', func.explode_outer(func.col('awards')))
                 .withColumn('awards', func.from_json(func.col('awards'), schema= schema_awards))
                 .withColumn('award_id', func.col('awards.id'))
                 .withColumn('award_funder_id', func.col('awards.funder_id'))
                 .withColumn('award_external_id', func.col('awards.funder_award_id'))
            .select(*[func.regexp_replace(func.col(id_col), 'https://openalex.org/','').alias(id_col) for id_col 
                in ['work_id','funder_id','award_id','award_funder_id'] ],
                    'award_external_id',
                    func.col('funders.display_name').alias('funder_name')
                   )
            )
works_funders.printSchema()


root
 |-- work_id: string (nullable = true)
 |-- funder_id: string (nullable = true)
 |-- award_id: string (nullable = true)
 |-- award_funder_id: string (nullable = true)
 |-- award_external_id: string (nullable = true)
 |-- funder_name: string (nullable = true)



In [29]:
works_funders.show()

+-----------+-----------+-----------+---------------+--------------------+--------------------+
|    work_id|  funder_id|   award_id|award_funder_id|   award_external_id|         funder_name|
+-----------+-----------+-----------+---------------+--------------------+--------------------+
|W1995237132|F4320307875|       NULL|           NULL|                NULL|Styrelsen för Int...|
|W1995237132|F4320323784|       NULL|           NULL|                NULL|Asian Institute o...|
|W1994831082|F4320322892|       NULL|           NULL|                NULL|Centre National d...|
|W2625982753|F4320315885|       NULL|           NULL|                NULL|Australian Govern...|
|W2626492834|F4320334704|G5905124557|    F4320337354|      1 T32 GM067555|Australian Resear...|
|W2626492834|F4320334704|G7158414504|    F4320337354|      1 T32 GM007185|Australian Resear...|
|W2626492834|F4320334704|G3383359617|    F4320334704|         FT160100233|Australian Resear...|
|W2626492834|F4320334704|G5099821281|   

In [30]:
works_funders.write.mode('overwrite').parquet("file:\\" + works_au_af_path.replace('works_au_af', 'works_funder'))

In [2]:
spark.catalog.clearCache()
spark.stop()